# ADNI

## INIT

In [32]:
from data_model.DataCleaner import DataCleaner
from dl_client import DatalakeClient
from tests_utils.manage_excel_support_file import *
import pandas as pd
import os

dataCleaner = DataCleaner(support_file_path='ADNI_variables_statistics.xlsx')
client = DatalakeClient()

# Download the raw files 
Exclusevily from ADNI dataset stored in the Datalake

In [33]:
file_codes = ['UCSDVOL']

In [34]:
search = client.query_files(
    query={'custom.level' : 'raw', 'custom.source' : 'ADNI', 'custom.file_code' : file_codes})

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True)

print(zip_files.keys())
print(len(zip_files.keys()))


dict_keys(['UCSDVOL_11Aug2025.csv'])
1


# Support file managment
operazione per popolare il file support file per i file considerati

In [35]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')

In [36]:
for file_name in list(zip_files.keys()):
    df = zip_files[file_name]
    infoSupportFile = InfoSupportFile(support_file, df, file_name)
    # delate the rows of the support file that are not in the df
    support_file, file_code = infoSupportFile.filter_variables()
    print(file_code)
    if file_code not in list(support_file['file_code']):
        print('not found in excel')
        continue
    # find the population variable code, and if not in support_file, add it
    pop, support_file = infoSupportFile.find_population_variable()
    # get the variable info and add it to the support_file    
    for key in df.keys():
        if key in support_file[support_file['file_code'] == file_code]['variable_code'].values:
            infoSupportFile.get_varible_info(key)
# save the updated support file
save_df(df_to_save=support_file, output_path=support_file_path)

UCSDVOL
UCSDVOL
PROBLEMA: nessuna chiave di popolazione trovata


In [37]:
new_support_file_name = 'ADNI_variables_cleaned1'
if os.path.isfile(new_support_file_name+'.xlsx'):
    update_new_support_file(support_file, new_support_file_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_support_file_name)

The ADNI_variables_cleaned1 file has been updated with the new file_code: ['UCSDVOL', nan, 'ADNI_DIAN_COMPARISON']
Open the file and verify it, if needed update the variables names and metadata


## IF SUPPORT FILE already populated

In [ ]:
support_file_path = 'ADNI_variables_statistics'
support_file = pd.read_excel(support_file_path+'.xlsx')

new_support_file_name = 'ADNI_variables_cleaned1'
if os.path.isfile(new_support_file_name+'.xlsx'):
    update_new_support_file(support_file, new_support_file_name, processed_file=file_codes)
else:
    create_new_support_file(support_file, support_file_path, new_name=new_support_file_name)


Open the new_support_file and fill in the new variable codes.

# FILE SPECIFIC DATA CLEANING 1


## UCSDVOL

In [38]:
file_code = 'UCSDVOL'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [39]:
# Important columns
columns_must_be_verified = ['BRAIN', 'EICV', 'VENTRICLES', 'LHIPPOC', 'RHIPPOC']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# Convert QCPASS values from 1/0 to 'complete'/'partial'
no_none_df = dataCleaner.convert_qcpass_values(no_none_df, col_name='QCPASS')
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='QCPASS')

Segmentation Status: 
 QCPASS
complete    2597
partial      531
Name: count, dtype: int64 
 <class 'str'>


In [40]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

Number of patients with duplicate dates:  0
Adopted visit selection strategy:
 Series([], Name: count, dtype: int64)


In [41]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [42]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

In [44]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [45]:
# optaining automatically info to save the file
lst_population = ['ADNI1']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [46]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## UCSF Longitudinal dataset

In [ ]:
file_codes = ['UCSFFSL51ALL', 'UCSFFSL51', 'UCSFFSL51Y1', 'UCSFFSL']
file_code = file_codes[3]

In [ ]:
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
if file_code == 'UCSFFSL': #the other UCSF longitudinal files have just partial immages segmentation
    no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
if file_code == 'UCSFFSL':
    lst_population = ['ADNI1','ADNIGO','ADNI2']
else:
    lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## UCSFFSX 
complete    4485\
partial        1\
hanno solo VISITCODE e non VISITCODE2 inoltre non hanno info sulla popolazione --> da inserire manualmente?

In [ ]:
file_code = 'UCSFFSX' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
######### ERRORE DA RISOLVERE
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)



infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1','ADNIGO','ADNI2']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## UCSFFX7

partial     11091\
complete      849

In [ ]:
file_code = 'UCSFFSX7' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') 

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

In [ ]:
# update the new info support file
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## UCSFFSX6
complete    2222\
partial       18

In [ ]:
file_code = 'UCSFFSX6' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS') ### farlo subito o poi?

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)



infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

In [ ]:
len(final_df['RID'].value_counts()[final_df['RID'].value_counts() == 1])

## UCSFFSX51

In [ ]:
file_code = 'UCSFFSX51' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# this file has no status column

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)


infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## UCSFFSX51_ADNI1_3T
partial    484


In [ ]:
file_code = 'UCSFFSX51_ADNI1_3T' 
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
# Important columns
columns_must_be_verified = ['ST37SV','ST10CV','ST24CV','ST26CV','ST29SV','ST40CV','ST96SV','ST83CV','ST85CV','ST88SV','ST99CV']

# Fixing unkown values and delating rows with no relevant data
no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
# this file has only partial segmentations in the STATUS ==> run funzione ma solo per verifica no modifica dataset... resterebbe vuoto? 
# VERIFICARE ma non da usare
test_df = dataCleaner.segmentation_complete_filter(no_none_df, filter_col='STATUS')

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
#Filtering variables
processed_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   

In [ ]:
# Assign new names to variables
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)


infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)

for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)


In [ ]:
# optaining automatically info to save the file
lst_population = ['ADNI1']
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

In [ ]:
# Upload the file into the Data Lake
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## ADNI MERGE

In [ ]:
file_code = 'ADNIMERGE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 


In [ ]:
# Important columns
columns_must_be_verified = ['APOE4', 'MMSE', 'Ventricles', 'Hippocampus', 'AGE']
single_column_required = ['DX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, single_column_required)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = datefix_df.copy(deep=True)
processed_df.rename(columns={'AGE': 'AGE_bl'}, inplace=True)
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['EXAMDATE'], age_bl=row['AGE_bl'], bl_date=row['EXAMDATE_bl']), axis=1)
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
processed_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')
processed_df = dataCleaner.categorize_diagnosis(processed_df, col_name='DX')
processed_df = dataCleaner.filter_variables(processed_df, list(zip_files.keys())[0], new_var=['AGE_bl', 'VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')   #in this case i might like to delate AGE_bl since it is now substituted by AGE


In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', processed_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## MMSE

In [ ]:
file_code = 'MMSE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'MMSE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True) 

In [ ]:
# Important columns
columns_must_be_verified = ['MMSCORE']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
# funzione VISCODE da examdate
final_df = dataCleaner.filter_variables(datefix_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')  

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')
    
result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## PTDEMOG

In [ ]:
file_code = 'PTDEMOG'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)
file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
columns_must_be_verified = ['PTGENDER', 'PTDOB', 'PTEDUCAT', 'PTETHCAT', 'PTRACCAT', 'PTADDX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['VISDATE'])
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'VISDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = datefix_df.copy(deep=True)
processed_df['AGE'] = processed_df.apply(lambda row: dataCleaner.add_calculated_age(exam_date=row['VISDATE'],birth_date=row['PTDOB'], birth_year=row['PTDOBYY']), axis=1)

In [ ]:
processed_df = dataCleaner.binarization_gender(processed_df, col_name='PTGENDER')
processed_df = dataCleaner.categorize_marry(processed_df, col_name='PTMARRY')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PTEDUCAT')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PTETHCAT')
final_df = dataCleaner.categorize_race(processed_df, col_name='PTRACCAT')

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['AGE', 'VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## ADSP_PHC_BIOMARKER

In [ ]:
file_code = 'ADSP_PHC_BIOMARKER'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'ADSP_PHC_BIOMARKER'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
columns_must_be_verified = ['PHC_Tau', 'PHC_pTau', 'PHC_AB42', 'AT_class']
single_column_required = ['PHC_Diagnosis']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, single_column_required)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['DRAWDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'DRAWDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
processed_df = dataCleaner.binarization_gender(datefix_df, col_name='PHC_Sex')
processed_df = dataCleaner.categorize_education(processed_df, col_name='PHC_Education')
processed_df = dataCleaner.categorize_ethnicity(processed_df, col_name='PHC_Ethnicity')
processed_df = dataCleaner.convert_to_two_bit(processed_df, col_name='AT_class')
final_df = dataCleaner.categorize_diagnosis(processed_df, col_name='PHC_Diagnosis')

In [ ]:
# non funziona la funzione perche black e Native Hawaian or PI sono invertite
mapping = {1: 1, 1.0: 1, 2: 2, 2.0: 2, 4: 3, 4.0: 3, 3: 4, 3.0: 4, 5: 5, 5.0: 5}
final_df['PHC_Race'] = final_df['PHC_Race'].map(mapping)

In [ ]:
final_df =dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## BLCHANGE

In [ ]:
file_code = 'BLCHANGE'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': 'BLCHANGE'
    }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [ ]:
columns_must_be_verified = ['BCMMSE', 'BCADAS', 'BCPREDX']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
final_df = dataCleaner.categorize_diagnosis(datefix_df, col_name='BCPREDX')

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)

## DXSUM

In [ ]:
file_code = 'DXSUM'
search = client.query_files(
    query={
        'custom.level' : 'raw',
        'custom.file_code': file_code
        }
)

zip_files = client.download_file(
    search['object_name'],  
    extract_zip=True
)

file_name = list(zip_files.keys())[0]
dataset = zip_files[file_name]

df_new = dataset.copy(deep=True)

In [ ]:
row_to_drop = df_new[df_new['VISCODE2'] == 'uns1'].index
df_new = df_new.drop(row_to_drop)

In [ ]:
columns_must_be_verified = ['DXNORM', 'DXMCI', 'DXNODEP']
required_column = ['DIAGNOSIS']

no_unknow_df = dataCleaner.replace_unknown_values(df_new)
no_none_df = dataCleaner.drop_if_all_none(no_unknow_df, columns_must_be_verified)
no_none_df = dataCleaner.drop_if_all_none(no_none_df, required_column)

In [ ]:
# Finding the exam code
datefix_df = dataCleaner.to_date_format(no_none_df, ['EXAMDATE']) 
# satranno tutte visit month 0 in quanto ha solo 1 visita per soggetto questo file
datefix_df = dataCleaner.find_exam_code(
    datefix_df, 
    date_column = 'EXAMDATE',
    viscode_refernce = 'VISCODE2', 
    patient_id_column = 'RID', 
    essential_variables= columns_must_be_verified)

In [ ]:
final_df = dataCleaner.categorize_diagnosis(datefix_df, col_name='DIAGNOSIS')
final_df = dataCleaner.to_date_format(final_df, ['EXAMDATE'])

In [ ]:
final_df = dataCleaner.filter_variables(final_df, list(zip_files.keys())[0], new_var=['VISIT_MONTH'], remove_var=['VISCODE', 'VISCODE2'], prefix='raw')

In [ ]:
new_support_file_path = 'ADNI_variables_cleaned1'
final_df, new_support_file = dataCleaner.new_variable_names(new_support_file_path+'.xlsx', final_df, file_code)

Now update the new_support_file to have an updated idea of the variable caratheristics and number of nan variables ==> which variables could be delated

In [ ]:
infoSupportFile = InfoSupportFile(new_support_file, final_df, file_name)
   
for key in final_df.keys():
    if key in new_support_file[new_support_file['file_code'] == file_code]['variable_code'].values:
        new_support_file = infoSupportFile.get_varible_info(key)

save_df(new_support_file, new_support_file_path)

In [ ]:
lst_population = infoSupportFile.get_present_populations()
new_file_name = dataCleaner.file_versions_name(file_name=file_name, suffix='_01')

result = client.upload_dataframe(
    df=final_df,
    object_name=new_file_name,
    prefix='cleaned/single_file/',
    metadata={
        'population':lst_population,
        'level': 'cleaned_01',
        'file_code': file_code,
        'source': 'ADNI'
    }
)